# Global steering analysis

Run the setup cell first. Review model, SAE, data paths, and run settings before starting. See the repository README for setup and pipeline order.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


## Final steering evaluation
These tables use the validation-selected checkpoint on the test and external datasets. Training curves are available in each run’s `epoch_metrics_long.csv`; they contain validation scores only.


In [ ]:
from pathlib import Path
import json
import pandas as pd

rows = []
for path in Path("data").glob("**/selected_checkpoint_metrics.csv"):
    config_path = path.parent / "config.json"
    config = json.loads(config_path.read_text())
    frame = pd.read_csv(path)
    frame["model"] = config["model_name"]
    frame["run_dir"] = str(path.parent)
    rows.append(frame)
if not rows:
    raise FileNotFoundError("Run train-steering with the train/validation/test protocol first.")
results = pd.concat(rows, ignore_index=True)
# Do not maximize test scores across runs. Select any run settings on validation.
display(results[["model", "run_dir", "dataset", "n", "accuracy", "macro_f1", "weighted_f1"]])
results.to_csv("data/steering_test_results.csv", index=False)
